# Load Predictive Models

In [1]:
import sys
sys.path.append('../utils/')
from nlp_utils import * 

df_train = open_data('../data/liar-plus/train2.tsv')
df_test = open_data('../data/liar-plus/test2.tsv')
df_val = open_data('../data/liar-plus/val2.tsv')
df_train.head()

df_train["statement"] = df_train["statement"].astype(str)
df_train.dropna(subset=['statement','label'], inplace=True)
df_train["statement"].apply(clean_text)
df_train = df_train[['statement','label']]

df_val["statement"] = df_val["statement"].astype(str)
df_val.dropna(subset=['statement','label'], inplace=True)
df_val["statement"].apply(clean_text)
df_val = df_val[['statement','label']]

df_test["statement"] = df_test["statement"].astype(str)
df_test.dropna(subset=['statement','label'], inplace=True)
df_test["statement"].apply(clean_text)
df_test = df_test[['statement','label']]

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import StandardScaler
import torch
import pandas as pd

spam_model_name = "mrm8488/bert-tiny-finetuned-sms-spam-detection"
spam_tokenizer = AutoTokenizer.from_pretrained(spam_model_name)
spam_model = AutoModelForSequenceClassification.from_pretrained(spam_model_name)
spam_model.eval()

def get_spam_scores(text_list, batch_size=16):
    scores = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        inputs = spam_tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=512
        )
        with torch.no_grad():
            outputs = spam_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)
            scores.extend(probs[:, 1].tolist())
    return scores

df_train['spam_score'] = get_spam_scores(df_train['statement'].tolist())
df_test['spam_score'] = get_spam_scores(df_test['statement'].tolist())
df_val['spam_score'] = get_spam_scores(df_val['statement'].tolist())

spam_scaler = StandardScaler()
df_train['spam_score'] = spam_scaler.fit_transform(df_train[['spam_score']])
df_val['spam_score'] = spam_scaler.transform(df_val[['spam_score']])   
df_test['spam_score'] = spam_scaler.transform(df_test[['spam_score']])

In [4]:
import spacy
# spacy.cli.download("en_core_web_md")
datum = df_train.iloc[0]
nlp = spacy.load("en_core_web_md")
doc = nlp(datum['statement'])
doc.vector.shape
statistic_types = {'CARDINAL', 'PERCENT', 'MONEY', 'QUANTITY'}

def stat_counter(text):
    if not isinstance(text, str):
        return 0
    doc = nlp(text)
    counter = 0
    for ent in doc.ents: 
        if ent.label_ in statistic_types:
            counter += 1
    return counter

from rapidfuzz import fuzz
conservative_bigrams = pd.read_csv('../data/top_conservative_bigrams.csv')['bigram']
liberal_bigrams = pd.read_csv('../data/top_liberal_bigrams.csv')['bigram']
def match_counter(statement, bigram_list, threshold):
    stat = nlp(str(statement))
    word = [word.text.lower() for word in stat]
    bigram_coll = [''.join(word[i:i+2]) for i in range(len(word)-1)]
    matches = 0
    for bigram in bigram_coll:
        for check in bigram_list:
            if fuzz.ratio(bigram, check) >= threshold:
                matches += 1
                break

    return matches


df_train['statistic_count'] = df_train['statement'].apply(stat_counter)
df_train['conservative_bigram_count'] = df_train['statement'].apply(
    lambda x: match_counter(x, conservative_bigrams, threshold=70)
)
df_train['liberal_bigram_count'] = df_train['statement'].apply(
    lambda x: match_counter(x, liberal_bigrams, threshold=70)
)

df_val['statistic_count'] = df_val['statement'].apply(stat_counter)
df_val['conservative_bigram_count'] = df_val['statement'].apply(
    lambda x: match_counter(x, conservative_bigrams, threshold=70)
)
df_val['liberal_bigram_count'] = df_val['statement'].apply(
    lambda x: match_counter(x, liberal_bigrams, threshold=70)
)

df_test['statistic_count'] = df_test['statement'].apply(stat_counter)
df_test['conservative_bigram_count'] = df_test['statement'].apply(
    lambda x: match_counter(x, conservative_bigrams, threshold=70)
)
df_test['liberal_bigram_count'] = df_test['statement'].apply(
    lambda x: match_counter(x, liberal_bigrams, threshold=70)
)

count_features = ["statistic_count", "conservative_bigram_count", "liberal_bigram_count"]
scaler_counts = StandardScaler()

df_train[count_features] = scaler_counts.fit_transform(df_train[count_features])
df_val[count_features]   = scaler_counts.transform(df_val[count_features])
df_test[count_features]  = scaler_counts.transform(df_test[count_features])

In [5]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def emotional_intensity_vader(text):
    if not isinstance(text, str):
        return 0.0
    if len(text) == 0:
        return 0.0
    vs = analyzer.polarity_scores(text)
    return abs(vs['compound'])

for df in [df_train, df_test, df_val]:
    df['emotional_intensity'] = df['statement'].apply(emotional_intensity_vader)

scaler_vader = StandardScaler()
df_train["emotional_intensity"] = scaler_vader.fit_transform(df_train[["emotional_intensity"]])
df_val["emotional_intensity"]   = scaler_vader.transform(df_val[["emotional_intensity"]])
df_test["emotional_intensity"]  = scaler_vader.transform(df_test[["emotional_intensity"]])

In [6]:
def add_veracity_features(df):
    df = df.copy()

    # ----- Spam feature -----
    df['spam_score'] = get_spam_scores(df['statement'].tolist())
    df['spam_score'] = spam_scaler.transform(df[['spam_score']])

    # ----- Count-based features -----
    df['statistic_count'] = df['statement'].apply(stat_counter)
    df['conservative_bigram_count'] = df['statement'].apply(
        lambda x: match_counter(x, conservative_bigrams, threshold=70)
    )
    df['liberal_bigram_count'] = df['statement'].apply(
        lambda x: match_counter(x, liberal_bigrams, threshold=70)
    )
    count_features = ["statistic_count", "conservative_bigram_count", "liberal_bigram_count"]
    df[count_features] = scaler_counts.transform(df[count_features])  # use transform, not fit_transform!

    # ----- Emotional intensity -----
    df['emotional_intensity'] = df['statement'].apply(emotional_intensity_vader)
    df['emotional_intensity'] = scaler_vader.transform(df[['emotional_intensity']])

    return df

In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_train['label'] = le.fit_transform(df_train['label'])
df_val['label'] = le.transform(df_val['label'])
df_test['label'] = le.transform(df_test['label'])

# Load and augment test_data

In [8]:
incoming_df = pd.read_csv('../data/labeled_articles.csv')

incoming_text = incoming_df[['id','text']]

incoming_scores = incoming_df[[c for c in incoming_df.columns if c not in set(['id','text'])]]

# GenAI

In [9]:
import pandas as pd
import os
from google import genai
from google.genai import types
import yaml

with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

api_key = config["gemini"]["api"]
client = genai.Client(api_key=api_key)

system_prompt = """You are an AI assistant assigned to evaluate the factuality of news statements using a generative fact-checking pipeline.

Your task is to analyze article text, incorporate predictive model outputs WITHOUT overweighting them, and compute factor scores using the scoring recipes below.

==========================
ANTI-BIAS CONSTRAINT
==========================
- Treat predictive model scores only as auxiliary context.
- Do NOT give these predictive scores undue weight.
- If your analysis of the article text contradicts the predictive scores, rely on the TEXTUAL EVIDENCE and explain the discrepancy.

==========================
FACTUALITY FACTORS (6 TOTAL)
==========================

1. AUTHENTICITY  
- Definition: Does the text present evidence that the claims are genuine, verifiable, and traceable?  
- Scoring Recipe (1–10): Look for verifiable details, named sources, timestamps, data, official statements; higher when concrete and falsifiable.  
- Output: numeric score + 1–2 sentences referencing evidence.

2. SENSATIONALISM  
- Definition: Presence of hyperbole, emotional language, exaggeration.  
- Scoring Recipe (1–10): Extract emotional/hyperbolic language, count dramatic constructions, score based on density and prominence.  
- Output: score + 2 example phrases.

3. POLITICAL BIAS  
- Definition: Degree to which the article leans left, center, or right.  
- Scoring Recipe (0–10 + tag): Identify partisan framing or selective omission.  
- Output: numeric score + category {{left, centrist, right, mixed}} + examples.

4. TOXICITY  
- Definition: Hostile, demeaning, or aggressive language.  
- Scoring Recipe (1–10): Identify insults, threats, aggression, and target.  
- Output: score + most toxic example phrase.

5. CONFIRMATION BIAS  
- Definition: Selective presentation of information reinforcing a preferred conclusion.  
- Scoring Recipe (1–10): Identify cherry-picked evidence or missing counterarguments.  
- Output: score + 1 example.

6. SHORT-TERM UTILITY (Profit Incentive)  
- Definition: Degree content maximizes clicks or engagement.  
- Scoring Recipe (1–10): Detect clickbait, urgent calls to action, monetization cues.  
- Output: score + 1–2 indicators of profit-driven framing.

==========================
VERACITY LABEL (REQUIRED)
==========================
- Output a final factuality classification: pants-fire, false, barely-true, half-true, mostly-true, true

==========================
OUTPUT FORMAT
==========================
- Final veracity label  
- Scores for all six factors  
- Short paragraph explaining WHY each score was assigned  
- Final combined veracity explanation  
- Structured JSON-like block per statement

==========================
REASONING FORMAT
==========================
- Provide concise, structured rationale per factor: key textual evidence, weighting of predictive scores, numeric reasoning (2–4 bullet points).  
- DO NOT reveal internal chain-of-thought or hidden reasoning.

==========================
EXAMPLES
==========================
Here are some examples of the expected output format and reasoning structure:
    - Example 1:
    {{
    "article": "A new study shows that drinking green tea daily reduces the risk of heart disease by 30%. The study surveyed 10,000 adults over 5 years and was published in the Journal of Cardiology.",
    "author": "Health Daily News",
    "authenticity": "9",    # The study is published in a reputable journal and includes a large sample size and clear methodology.
    "sensationalism": "3",  # Mild exaggeration in phrasing 'reduces the risk by 30%', but mostly factual.
    "political bias": "1",  # No political framing detected.
    "toxicity": "1",    # No hostile or demeaning language.
    "confirmation bias": "2",   # Some emphasis on positive findings without mention of limitations, but minimal.
    "short-term utility": "2",  # Headline is slightly attention-grabbing, but content is largely informational.
    }}

    - Example 2:
    {{
        "article": "Politician X is the worst leader in history! Everything they touch fails, and the economy is collapsing under their rule.",
        "author": "Partisan Weekly",
        "authenticity": "2",    # Claims are vague and unsupported; no verifiable evidence is cited.
        "sensationalism": "9",  # Highly emotional and hyperbolic language such as 'worst leader' and 'everything they touch fails'.
        "political bias": "10", # Strong right/left framing (depending on context); clearly targeting a political figure.
        "toxicity": "8",    # Personal attacks and extreme language directed at the politician.
        "confirmation bias": "8",   # Selectively presents negative information; ignores any positive actions or counterarguments.
        "short-term utility": "7",  # Designed to provoke outrage and drive engagement.
    }}

    - Example 3:
    {{
        "article": "Local bakery wins award for best chocolate cake. The contest included 50 bakeries, and judges highlighted creativity and flavor balance.",
        "author": "Town Gazette",
        "authenticity": "8",    # Contest results are verifiable; named judges and participants are listed.
        "sensationalism": "2",  # Mildly positive tone but not exaggerated.
        "political bias": "1",  # No political content.
        "toxicity": "1",    # No toxic language present.
        "confirmation bias": "1",   # Balanced reporting; no selective framing detected.
        "short-term utility": "3",  # Lightly engaging human-interest story.
    }}

    - Example 4:
    {{
        "article": "Experts warn that a major cyberattack could hit the US next month. Anonymous sources say the threat is imminent.",
        "author": "TechAlert News",
        "authenticity": "3", # Based on anonymous sources; lacks verifiable evidence.
        "sensationalism": "8", # Phrases like 'major cyberattack' and 'imminent' are highly alarming.
        "political bias": "2", # No clear political slant, mostly technical framing.
        "toxicity": "1", # No hostile language.
        "confirmation bias": "5", # Focuses on worst-case scenarios without context or probability discussion.
        "short-term utility": "8", # Headline and framing likely to generate clicks and engagement.
    }}
"""

user_prompt = f""""Here is the dataset to evaluate:
        {incoming_text}
Here are the predictive model outputs:
        {incoming_scores}
Begin the factor scoring and veracity prediction.
"""

response = client.models.generate_content(
    model="gemini-2.5-pro",
    config=types.GenerateContentConfig(
        system_instruction=system_prompt),
    contents=user_prompt
)

print(response.text)

**Veracity Label:** false
**Authenticity:** 1/10
The premise of a mayor initiating a "city-wide manhunt" for people who use a common workplace phrase is absurd and not verifiable as a real event. This is a satirical piece, and the claims are fictional.

**Sensationalism:** 8/10
The article uses hyperbolic and exaggerated language for comedic effect. Phrases like "city-wide manhunt" and "ruthlessly hunted down" are intentionally over-the-top to create a sensational narrative.

**Political Bias:** 7/10 {mixed}
The article is political satire, targeting a real-life Democratic Socialist politician, Zohran Mamdani. It satirizes a progressive political figure by portraying his supposed policy as absurdly authoritarian, which can appeal to a right-leaning or anti-authoritarian audience.

**Toxicity:** 7/10
The language is aggressive and demeaning in a satirical context. Describing the targets as "insipid nine-to-fivers" who will be "ruthlessly hunted down" constitutes hostile framing.

**Conf

In [10]:
import json
import re

def extract_json_dicts(text):
    """
    Extracts all JSON objects from the given text and returns them as Python dictionaries.
    """
    # Regex to match JSON blocks inside ```json ... ```
    code_block_pattern = r"```json\s*(\{.*?\})\s*```"
    
    # Find all JSON snippets
    json_strings = re.findall(code_block_pattern, text, flags=re.DOTALL)

    dicts = []
    for js in json_strings:
        try:
            parsed = json.loads(js)
            dicts.append(parsed)
        except json.JSONDecodeError as e:
            print("JSON decode error:", e)
            print("Offending JSON:", js[:200], "...")
    
    return dicts

llm_outputs = extract_json_dicts(response.text)
llm_outputs

[{'id': 1,
  'text': 'NEW YORK, NY — Mayor-elect Zohran Mamdani announced Friday that upon taking office, he would initiate a city-wide manhunt for anyone who has ever said the phrase, “Is it Friday yet?”...',
  'final_veracity_label': 'false',
  'factuality_factors': {'authenticity': {'score': 1,
    'reasoning': "The claim of a 'city-wide manhunt' for a common phrase is patently absurd and fictional. The predictive model score of 2 is consistent with this analysis, as the text presents no verifiable or believable evidence."},
   'sensationalism': {'score': 8,
    'reasoning': "The text uses extreme hyperbole for satirical effect. Phrases such as 'city-wide manhunt' and 'ruthlessly hunted down' are intentionally sensational. The model's score of 7 aligns with this assessment."},
   'political_bias': {'score': 7,
    'category': 'mixed',
    'reasoning': "The article is political satire targeting a real left-wing politician, Zohran Mamdani. It uses caricature to mock a political ideolo

In [11]:
pd.DataFrame(llm_outputs)

,id,text,final_veracity_label,factuality_factors,veracity_explanation
0,1,"NEW YORK, NY — Mayor-elect Zohran Mamdani anno...",false,"{'authenticity': {'score': 1, 'reasoning': 'Th...",The article is satirical and its central claim...
1,2,Walmart is proving to be America’s antidote to...,half-true,"{'authenticity': {'score': 7, 'reasoning': 'Th...","The article combines factual data, such as Wal..."
2,3,LAS VEGAS—Shaking his head in frustration afte...,false,"{'authenticity': {'score': 1, 'reasoning': 'Th...",This is a satirical article. The described eve...
3,4,Nov 20 (Reuters) - Studies from Novo Nordisk (...,true,"{'authenticity': {'score': 9, 'reasoning': 'Th...",The statement is a factual report from a relia...
4,5,The Buffalo Bills and Houston Texans will meet...,false,"{'authenticity': {'score': 2, 'reasoning': 'Th...","This statement is a prediction, not a statemen..."
5,6,"For the first time in his second term, Preside...",true,"{'authenticity': {'score': 9, 'reasoning': 'Th...","This statement describes a factual, verifiable..."
6,7,While hosting Saudi Crown Prince Mohammed bin ...,false,"{'authenticity': {'score': 1, 'reasoning': 'Th...",This statement is a piece of political satire ...
7,8,SAN DIEGO — San Diego homeowners looking to se...,mostly-true,"{'authenticity': {'score': 7, 'reasoning': 'Th...",The statement describes a common economic situ...
8,9,The Democrat is accused of stealing Federal Em...,true,"{'authenticity': {'score': 8, 'reasoning': 'Th...",The statement reports that an individual is 'a...
9,10,Lawmakers Fear Brazil Will Fast Track Bill ‘Ta...,barely-true,"{'authenticity': {'score': 5, 'reasoning': 'Th...",The statement's truthfulness rests on whether ...
